In [1]:
from IPython.display import HTML
HTML("""
<style>
/* Hide code inputs in Jupyter Book pages */
div.cell_input {display:none;}
</style>
""")

## ICESEE GUI

In [2]:
import ipywidgets as W
from IPython.display import display, HTML, clear_output
from pathlib import Path
import datetime, os, sys, shutil, subprocess

# ---- UI bits ----
header = W.HTML("""
<div style="padding:10px 0 6px 0">
  <div style="font-size:26px;font-weight:700; line-height:1.2;"> </div>
  <div style="color:#555; margin-top:4px;">
    Run ICESEE examples with one click. Outputs and reports are saved to the wrapper and shown below.
  </div>
</div>
""")

example = W.Dropdown(
    options=[
        ("Lorenz-96 (fully runnable)", "lorenz96"),
        ("Flowline (template)", "flowline"),
        ("ISSM (template)", "issm"),
        ("Icepack (template)", "icepack"),
    ],
    value="lorenz96",
    description="Example:",
    style={"description_width": "90px"},
)

preset = W.Dropdown(
    options=[("Fast", "fast"), ("Default", "default"), ("Robust", "robust")],
    value="default",
    description="Preset:",
    style={"description_width": "90px"},
)

filter_type = W.Dropdown(
    options=[("True/Wrong (demo)", "true-wrong"), ("EnKF (generic)", "enkf")],
    value="true-wrong",
    description="Filter:",
    style={"description_width": "90px"},
)

n_ens = W.IntSlider(value=20, min=5, max=200, step=5, description="Ens:",
                    style={"description_width":"90px"}, continuous_update=False)

seed = W.IntText(value=42, description="Seed:", style={"description_width":"90px"})

postprocess = W.Checkbox(value=True, description="Generate report (read_results) after run")
open_latest = W.Checkbox(value=True, description="After run: open Latest Run dashboard")

run_btn = W.Button(description="Run", button_style="success", icon="play")
clear_btn = W.Button(description="Clear", button_style="", icon="trash")

status = W.HTML("<span style='padding:4px 10px;border-radius:999px;background:#eee;color:#444'>Idle</span>")
log = W.Output(layout={"border":"1px solid #e6e6e6", "padding":"10px", "border_radius":"10px", "max_height":"260px", "overflow":"auto"})
preview = W.Output(layout={"border":"1px solid #e6e6e6", "padding":"10px", "border_radius":"10px"})

controls = W.VBox(
    [
        W.HTML("<b>Run settings</b>"),
        example,
        preset,
        filter_type,
        n_ens,
        seed,
        postprocess,
        open_latest,
        W.HBox([run_btn, clear_btn]),
        W.HTML("<div style='margin-top:6px'><b>Status:</b></div>"),
        status,
    ],
    layout=W.Layout(gap="10px")
)

right = W.VBox(
    [
        W.HTML("<b>Run log</b>"),
        log,
        W.HTML("<b style='margin-top:6px'>Results preview</b>"),
        preview
    ],
    layout=W.Layout(gap="10px")
)

page = W.VBox(
    [
        header,
        W.HBox([controls, right], layout=W.Layout(gap="18px"))
    ]
)

display(page)

In [3]:
# backend bits
import papermill as pm
from IPython.display import Image

def find_repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "external" / "ICESEE").exists() and (p / "icesee_jupyter_book").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not locate repo root containing external/ICESEE and icesee_jupyter_book.")

def ensure_external_on_path(root: Path):
    ext = (root / "external").resolve()
    if str(ext) not in sys.path:
        sys.path.insert(0, str(ext))
    os.environ["PYTHONPATH"] = f"{ext}:{os.environ.get('PYTHONPATH','')}"
    return ext

def make_run_dir(root: Path, example_key: str):
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = root / "icesee_jupyter_book" / "_static" / "generated" / example_key / stamp
    (run_dir / "results").mkdir(parents=True, exist_ok=True)
    (run_dir / "figures").mkdir(parents=True, exist_ok=True)
    return run_dir

EXAMPLES = {
    "lorenz96": {
        "script": Path("applications/lorenz_model/examples/lorenz96/run_da_lorenz96.py"),
        "report_nb": Path("applications/lorenz_model/examples/lorenz96/read_results.ipynb"),
        "assets": ["_modelrun_datasets"],   # things report expects
        "model_name": "lorenz",
    },
    # templates (fill later)
    "flowline": {"script": None, "report_nb": None, "assets": [], "model_name": "flowline"},
    "issm": {"script": None, "report_nb": None, "assets": [], "model_name": "issm"},
    "icepack": {"script": None, "report_nb": None, "assets": [], "model_name": "icepack"},
}

def expected_h5_name(filter_type_value: str, model_name: str) -> str:
    return f"{filter_type_value}-{model_name}.h5"

def collect_expected_h5(root: Path, run_dir: Path, example_key: str, filter_type_value: str):
    model_name = EXAMPLES[example_key]["model_name"]
    h5_name = expected_h5_name(filter_type_value, model_name=model_name)
    target_h5 = run_dir / "results" / h5_name

    if target_h5.exists():
        return target_h5

    # Search under external/ICESEE first (fast + relevant)
    candidates = list((root / "external" / "ICESEE").glob(f"**/results/{h5_name}"))
    if not candidates:
        candidates = list(root.glob(f"**/results/{h5_name}"))
    if not candidates:
        raise FileNotFoundError(f"Could not find produced H5 '{h5_name}' anywhere under repo.")

    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    src_h5 = candidates[0]
    target_h5.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_h5, target_h5)

    # Copy any figures sitting next to where results were produced
    src_fig_dir = src_h5.parent.parent / "figures"
    if src_fig_dir.exists():
        for f in src_fig_dir.glob("*.png"):
            shutil.copy2(f, run_dir / "figures" / f.name)

    return target_h5

def mirror_assets_for_report(root: Path, run_dir: Path, example_key: str):
    example_dir = (root / "external" / "ICESEE" / EXAMPLES[example_key]["script"].parent).resolve()
    for a in EXAMPLES[example_key]["assets"]:
        src = example_dir / a
        if src.exists():
            dst = run_dir / a
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

def run_example_script(root: Path, run_dir: Path, example_key: str, filter_type_value: str, n_ens_value: int, seed_value: int):
    ensure_external_on_path(root)

    meta = EXAMPLES[example_key]
    if meta["script"] is None:
        raise RuntimeError(f"Example '{example_key}' is not runnable here yet (template only).")

    script = (root / "external" / "ICESEE" / meta["script"]).resolve()
    if not script.exists():
        raise FileNotFoundError(f"Cannot find script: {script}")

    env = os.environ.copy()
    env["ICESEE_FILTER_TYPE"] = str(filter_type_value)
    env["ICESEE_NENS"] = str(n_ens_value)
    env["ICESEE_SEED"] = str(seed_value)
    env["ICESEE_OUTDIR"] = str(run_dir)
    env["ICESEE_RESULTS_DIR"] = str(run_dir / "results")
    env["ICESEE_FIG_DIR"] = str(run_dir / "figures")

    cmd = [sys.executable, str(script)]
    p = subprocess.run(cmd, cwd=str(run_dir), env=env, capture_output=True, text=True)
    return p

def run_report(root: Path, run_dir: Path, example_key: str):
    meta = EXAMPLES[example_key]
    if meta["report_nb"] is None:
        return None

    nb_in = (root / "external" / "ICESEE" / meta["report_nb"]).resolve()
    if not nb_in.exists():
        raise FileNotFoundError(f"Report notebook not found: {nb_in}")

    nb_out = run_dir / "report.ipynb"

    # Make sure local assets exist in run_dir for relative loads
    mirror_assets_for_report(root, run_dir, example_key)

    pm.execute_notebook(
        input_path=str(nb_in),
        output_path=str(nb_out),
        cwd=str(run_dir),
        log_output=True,
    )
    return nb_out

def render_preview(run_dir: Path):
    with preview:
        clear_output()
        h5s = sorted((run_dir / "results").glob("*.h5"))
        pngs = sorted((run_dir / "figures").glob("*.png"))

        display(HTML(f"<div><b>Run folder:</b> <code>{run_dir}</code></div>"))
        display(HTML(f"<div style='margin-top:6px'><b>Results:</b> {len(h5s)} H5, {len(pngs)} PNG</div>"))

        if h5s:
            items = "".join([f"<li><code>{p.name}</code></li>" for p in h5s[:10]])
            display(HTML(f"<ul>{items}</ul>"))

        if pngs:
            display(HTML("<hr><b>Figures</b>"))
            for p in pngs[:8]:
                display(Image(filename=str(p)))
        else:
            display(HTML("<div style='color:#777;margin-top:8px'>No figures found yet.</div>"))

        # Link hint
        report_nb = run_dir / "report.ipynb"
        if report_nb.exists():
            display(HTML("<hr><b>Report</b>"))
            display(HTML(f"<div><code>{report_nb.name}</code> saved in run folder.</div>"))

In [4]:
## Wire the buttons (call backs) ##
def set_status(text, kind="idle"):
    colors = {
        "idle": ("#eee", "#444"),
        "running": ("#fff3cd", "#7a5d00"),
        "ok": ("#d1e7dd", "#0f5132"),
        "bad": ("#f8d7da", "#842029"),
    }
    bg, fg = colors.get(kind, ("#eee", "#444"))
    status.value = f"<span style='padding:4px 10px;border-radius:999px;background:{bg};color:{fg}'>{text}</span>"

def apply_preset():
    p = preset.value
    if p == "fast":
        n_ens.value = 10
        seed.value = 1
        filter_type.value = "true-wrong"
    elif p == "default":
        n_ens.value = 20
        seed.value = 42
        filter_type.value = "true-wrong"
    elif p == "robust":
        n_ens.value = 80
        seed.value = 42
        filter_type.value = "true-wrong"

preset.observe(lambda c: apply_preset(), names="value")
apply_preset()

LATEST_PTR = None  # store last run folder in memory

def on_clear(_):
    with log:
        clear_output()
    with preview:
        clear_output()
    set_status("Idle", "idle")

def on_run(_):
    global LATEST_PTR
    root = find_repo_root()

    set_status("Running…", "running")
    with log:
        clear_output()
        print("[ICESEE-OnLINE] example:", example.value)
        print("[ICESEE-OnLINE] preset:", preset.value)
        print("[ICESEE-OnLINE] filter:", filter_type.value, "n_ens:", n_ens.value, "seed:", seed.value)

        try:
            run_dir = make_run_dir(root, example.value)
            print("[ICESEE-OnLINE] run_dir:", run_dir)

            p = run_example_script(root, run_dir, example.value, filter_type.value, n_ens.value, seed.value)

            if p.stdout:
                print("\n--- STDOUT (tail) ---")
                print(p.stdout[-4000:])
            if p.stderr:
                print("\n--- STDERR (tail) ---")
                print(p.stderr[-4000:])

            if p.returncode != 0:
                set_status("Failed", "bad")
                print("\n[ICESEE-OnLINE] Run failed.")
                return

            # Normalize outputs for wrapper/report
            print("\n[ICESEE-OnLINE] Collecting expected outputs into wrapper…")
            h5 = collect_expected_h5(root, run_dir, example.value, filter_type.value)
            print("[ICESEE-OnLINE] H5 ready:", h5)

            # Report
            if postprocess.value:
                print("\n[ICESEE-OnLINE] Generating report…")
                rpt = run_report(root, run_dir, example.value)
                if rpt:
                    print("[ICESEE-OnLINE] Report written:", rpt)

            LATEST_PTR = run_dir
            set_status("Complete", "ok")
            render_preview(run_dir)

            if open_latest.value:
                print("\nTip: open the 'Latest Run' page in the sidebar to view the newest run.")

        except Exception as e:
            set_status("Error", "bad")
            print("\n[ICESEE-OnLINE] ERROR:", e)

run_btn.on_click(on_run)
clear_btn.on_click(on_clear)